In [0]:
from pyspark.sql import functions as F, Window

SILVER = "workspace.riesgo_silver"
BRONZE = "workspace.riesgo_bronze"

def guardar_silver(con_motivo, tabla: str, claves: list[str]):
    """Separa válidos y rechazados, deduplica por `claves` y escribe en Silver."""
    validos = con_motivo.filter(F.col("_motivo_rechazo").isNull()).drop("_motivo_rechazo")
    rechazados = con_motivo.filter(F.col("_motivo_rechazo").isNotNull())

    orden = Window.partitionBy(*claves).orderBy("_ingerido_en", "_archivo_origen")
    unicos = validos.withColumn("_n", F.row_number().over(orden)).filter("_n = 1").drop("_n")

    print(f"{tabla}: bronze={con_motivo.count()} rechazados={rechazados.count()} "
          f"duplicados={validos.count() - unicos.count()} a_silver={unicos.count()}")

    unicos.createOrReplaceTempView(f"{tabla}_nuevos")
    spark.sql(f"CREATE TABLE IF NOT EXISTS {SILVER}.{tabla} AS SELECT * FROM {tabla}_nuevos LIMIT 0")
    condicion = " AND ".join(f"d.{c} = n.{c}" for c in claves)
    spark.sql(f"""
        MERGE INTO {SILVER}.{tabla} AS d
        USING {tabla}_nuevos AS n
        ON {condicion}
        WHEN NOT MATCHED THEN INSERT *
    """)
    (rechazados.write.mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER}.{tabla}_cuarentena"))


PRESTAMOS

In [0]:
tipado = spark.table(f"{BRONZE}.prestamos").select(
    "prestamo_id", "solicitud_id", "cliente_id",
    F.upper(F.trim("producto")).alias("producto"),
    F.expr("try_cast(fecha_desembolso as date)").alias("fecha_desembolso"),
    F.expr("try_cast(monto_desembolsado as decimal(18,2))").alias("monto_desembolsado"),
    F.expr("try_cast(tasa_mensual as decimal(6,4))").alias("tasa_mensual"),
    F.expr("try_cast(plazo_meses as int)").alias("plazo_meses"),
    F.expr("try_cast(cuota_mensual as decimal(18,2))").alias("cuota_mensual"),
    "_archivo_origen", "_ingerido_en",
)

con_motivo = tipado.withColumn(
    "_motivo_rechazo",
    F.when(F.col("prestamo_id").isNull(), "SIN_ID")
     .when(F.col("fecha_desembolso").isNull(), "FECHA_INVALIDA")
     .when(F.col("monto_desembolsado").isNull() | (F.col("monto_desembolsado") <= 0), "MONTO_INVALIDO")
     .when(F.col("plazo_meses").isNull() | (F.col("plazo_meses") < 1), "PLAZO_INVALIDO")
     .when(F.col("tasa_mensual").isNull() | (F.col("tasa_mensual") <= 0) | (F.col("tasa_mensual") > 0.05),
           "TASA_FUERA_DE_RANGO"),
)

guardar_silver(con_motivo, "prestamos", ["prestamo_id"])


prestamos: bronze=1554 rechazados=4 duplicados=0 a_silver=1550


PAGOS

In [0]:
prestamos_validos = (spark.table(f"{SILVER}.prestamos")
    .select("prestamo_id").withColumn("_existe", F.lit(True)))

tipado = spark.table(f"{BRONZE}.pagos").select(
    "pago_id", "prestamo_id",
    F.expr("try_cast(numero_cuota as int)").alias("numero_cuota"),
    F.expr("try_cast(fecha_pago as date)").alias("fecha_pago"),
    F.expr("try_cast(monto_pagado as decimal(18,2))").alias("monto_pagado"),
    F.upper(F.trim("medio_pago")).alias("medio_pago"),
    "_archivo_origen", "_ingerido_en",
)

con_motivo = (tipado
    .join(prestamos_validos, "prestamo_id", "left")
    .select(*tipado.columns, "_existe")
    .withColumn(
        "_motivo_rechazo",
        F.when(F.col("pago_id").isNull(), "SIN_ID")
         .when(F.col("fecha_pago").isNull(), "FECHA_INVALIDA")
         .when(F.col("monto_pagado").isNull() | (F.col("monto_pagado") <= 0), "MONTO_INVALIDO")
         .when(F.col("numero_cuota").isNull() | (F.col("numero_cuota") < 1), "CUOTA_INVALIDA")
         .when(F.col("_existe").isNull(), "PRESTAMO_INEXISTENTE"))
    .drop("_existe"))

guardar_silver(con_motivo, "pagos", ["pago_id"])


pagos: bronze=1491 rechazados=5 duplicados=15 a_silver=1471


solicitudes

In [0]:
tipado = spark.table(f"{BRONZE}.solicitudes").select(
    "solicitud_id", "cliente_id",
    F.expr("try_cast(fecha_solicitud as date)").alias("fecha_solicitud"),
    F.upper(F.trim("producto")).alias("producto"),
    F.upper(F.trim("canal")).alias("canal"),
    F.expr("try_cast(monto_solicitado as decimal(18,2))").alias("monto_solicitado"),
    F.expr("try_cast(plazo_meses as int)").alias("plazo_meses"),
    F.expr("try_cast(score_buro as int)").alias("score_buro"),
    F.upper(F.trim("estado")).alias("estado"),
    F.upper(F.trim("motivo_rechazo")).alias("motivo_rechazo"),
    "_archivo_origen", "_ingerido_en",
)

con_motivo = tipado.withColumn(
    "_motivo_rechazo",
    F.when(F.col("solicitud_id").isNull(), "SIN_ID")
     .when(F.col("fecha_solicitud").isNull(), "FECHA_INVALIDA")
     .when(~F.col("producto").isin("LI", "CC", "MC"), "PRODUCTO_INVALIDO")
     .when(~F.col("canal").isin("APP", "WEB", "ALIADO", "REFERIDO"), "CANAL_INVALIDO")
     .when(F.col("monto_solicitado").isNull() | (F.col("monto_solicitado") <= 0), "MONTO_INVALIDO")
     .when(~F.col("estado").isin("APROBADA", "RECHAZADA"), "ESTADO_INVALIDO"),
)

guardar_silver(con_motivo, "solicitudes", ["solicitud_id"])


solicitudes: bronze=3641 rechazados=7 duplicados=16 a_silver=3618


clientes

In [0]:
tipado = spark.table(f"{BRONZE}.clientes").select(
    "cliente_id",
    F.trim("ciudad").alias("ciudad"),
    F.upper(F.trim("ocupacion")).alias("ocupacion"),
    F.expr("try_cast(ingreso_mensual as decimal(18,2))").alias("ingreso_mensual"),
    F.expr("try_cast(score_buro as int)").alias("score_buro"),
    F.expr("try_cast(fecha_actualizacion as date)").alias("fecha_actualizacion"),
    "_archivo_origen", "_ingerido_en",
)

con_motivo = tipado.withColumn(
    "_motivo_rechazo",
    F.when(F.col("cliente_id").isNull(), "SIN_ID")
     .when(F.col("fecha_actualizacion").isNull(), "FECHA_INVALIDA")
     .when(F.col("score_buro").isNull() | ~F.col("score_buro").between(300, 850), "SCORE_FUERA_DE_RANGO")
     .when(F.col("ingreso_mensual").isNull() | (F.col("ingreso_mensual") <= 0), "INGRESO_INVALIDO"),
)

guardar_silver(con_motivo, "clientes", ["cliente_id", "fecha_actualizacion"])


clientes: bronze=2806 rechazados=2 duplicados=0 a_silver=2804
